# 01 · Ingestion pipeline
**What happens when a faculty member uploads a file:** parse → chunk → embed → store in Chroma.
Each step is shown separately, then run end-to-end exactly as the API does. Data written here is reused by notebooks 02–05.

In [ ]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
from dotenv import load_dotenv
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# Automatically load environment variables from rag_service/.env and repo-root .env
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / ".env", override=False)

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env (default when keys are present)
has_keys = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"))
OFFLINE = (os.getenv("NB_OFFLINE", "0" if has_keys else "1") == "1")
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings, reset_settings_cache
reset_settings_cache()
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

## 1. Create synthetic course files (PDF, PPTX, DOCX, XLSX, MD, CSV, HTML)

In [ ]:
import synthetic
files = synthetic.make_all()
for subj, fs in files.items():
    print(subj, {n: f"{len(b)//1024 or 1} KB" for n, b in fs.items()})

## 2. Parse (page/slide/sheet aware)

In [ ]:
from app.parsers import parse_bytes
secs = parse_bytes(files["BIO101"]["bio_notes.pdf"], "bio_notes.pdf")
for s in secs: print(f"page {s.page}:", s.text[:110].replace("\n"," "), "...")

## 3. Chunk (recursive splitter, provenance metadata, stable ids)

In [ ]:
from app.chunker import chunk_sections
chunks = chunk_sections(secs, "demo-doc", {"subject_code":"BIO101","is_active":True,"file_name":"bio_notes.pdf"})
print(len(chunks), "chunks | sizes:", [len(c.page_content) for c in chunks])
print(json.dumps(chunks[0].metadata, indent=1))

## 4. Embed

In [ ]:
from app.providers import get_embeddings
vecs = get_embeddings().embed_documents([c.page_content for c in chunks])
print("vectors:", len(vecs), "x", len(vecs[0]))

## 5. Run the full pipeline for every synthetic file (this is what `POST /ingest` does)

In [ ]:
from app.ingest import ingest_document, get_status
results = {}
for subj, fs in files.items():
    for name, data in fs.items():
        did = f"{subj}-{name}"
        results[did] = ingest_document(did, {"subject": subj, "is_active": True}, data=data, file_name=name)
import pandas as pd
pd.DataFrame(results).T[["chunkCount","durationMs","collection"]]

## 6. Verify what is really stored

In [ ]:
from app.store import get_store
st = get_store()
print("total chunks in collection:", st.count())
print(get_status("BIO101-bio_notes.pdf"))
sample = st.chunks_of("PHY101-physics.docx")[0]
print(sample.metadata["subject_code"], "| page", sample.metadata["page"], "|", sample.page_content[:100])

## 7. Idempotency + failure handling

In [ ]:
n0 = len(st.chunks_of("PHY101-physics.docx"))
ingest_document("PHY101-physics.docx", {"subject":"PHY101","is_active":True}, data=files["PHY101"]["physics.docx"], file_name="physics.docx")
assert len(st.chunks_of("PHY101-physics.docx")) == n0, "re-ingest must not duplicate"
from app.parsers import ParseError
try: ingest_document("bad", {"subject":"X"}, data=b"junk", file_name="bad.pdf")
except ParseError as e: print("clean failure ->", e, "| status:", get_status("bad")["status"])